# A first look at Gaia data

In this notebook we work with real data from Gaia Data Release 3 (DR3, released June 2022).
We will:

1. Query the Gaia archive and load a table of all well-measured stars within 50 parsecs
2. Use `pandas` to explore the table
3. Find our nearest neighbours, including a few famous ones
4. Make a map of the sky and a Hertzsprung-Russell diagram
5. Look at where the known exoplanets are
6. Look at the binary star orbits Gaia has already measured, and see where the planets would be

Like before, **Try it** cells have blanks (`____`) to fill in. Later cells often use the result of a
**Try it** cell, so get each one working before moving on. If you're stuck for more than a few
minutes, ask a mentor.

There's a lot of new `pandas` code in this notebook. You don't need to memorize it. Each new piece
is explained the first time it appears, and there's a short summary at the end.

In [ ]:
# Setup. If the data files aren't here (for example on Google Colab), download them.
# On Colab, also install astroquery. Elsewhere this cell does nothing.
import os, sys, subprocess, urllib.request

DATA_DIR = "data/"
DATA_URL = "https://raw.githubusercontent.com/vedantchandra/srmp-gaia/main/tutorial/data/"
os.makedirs(DATA_DIR, exist_ok=True)
for name in ["gaia_50pc.csv", "exoplanets.csv", "gaia_dr3_orbits_100pc.csv"]:
    if not os.path.exists(DATA_DIR + name):
        print("downloading", name)
        urllib.request.urlretrieve(DATA_URL + name, DATA_DIR + name)

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "astroquery"])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord

## 1. Getting the data

Gaia's catalogs live in the **Gaia archive** (https://gea.esac.esa.int/archive/). The main table,
`gaiadr3.gaia_source`, has one row for each of 1.8 billion sources and about 150 columns.
That's far too much to download, so we ask the archive for just the rows and columns we want.

The archive understands a language called **ADQL**, which is a version of SQL (a standard
language for querying databases). A query looks like this:

```sql
SELECT <columns> FROM <table> WHERE <conditions>
```

The `astroquery` package sends the query for us and gives back a table. Here's a tiny one: all
stars with a parallax above 300 mas, so closer than about 3.3 pc.

In [ ]:
from astroquery.gaia import Gaia

query = '''
SELECT source_id, ra, dec, parallax, phot_g_mean_mag
FROM gaiadr3.gaia_source
WHERE parallax > 300
'''

try:
    result = Gaia.launch_job(query).get_results()
    print(result)
except Exception as e:
    print("Could not reach the Gaia archive. That's OK, we'll use saved data below.")
    print(e)

If that worked, you just searched a catalog of 1.8 billion stars, and found about ten.
(You may also see a message about "breaking changes" in the archive. That's a notice for
programmers, not an error. You can ignore it.)

Some of the columns:

| Column | Meaning | Units |
|---|---|---|
| `source_id` | Unique ID number for each source | |
| `ra`, `dec` | Right ascension and declination: position on the sky | degrees |
| `parallax` | Parallax $\varpi$ | mas |
| `parallax_error` | Uncertainty on the parallax | mas |
| `pmra`, `pmdec` | Proper motion: how fast the star moves across the sky, in the RA and Dec directions. Written $\mu_{\alpha*}$ and $\mu_\delta$ in equations | mas/yr |
| `phot_g_mean_mag` | Brightness in Gaia's G band (a broad optical filter). Magnitudes are backwards: bigger = fainter | mag |
| `bp_rp` | Colour: brightness difference between Gaia's blue (BP) and red (RP) filters. Bigger = redder (cooler) | mag |
| `ruwe` | "Renormalised unit weight error": how far the star's measured positions are from the best simple model of a single star moving in a straight line (plus parallax). About 1 is normal. Much bigger than 1 means the simple model doesn't fit | |

The full list is in the Gaia DR3 documentation
(https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_main_source_catalogue/ssec_dm_gaia_source.html).

`ruwe` will matter a lot for our project. A star that wobbles because of a companion doesn't fit
the single-star model well, so it gets a high RUWE. (Roughly: RUWE measures how far the data points
sit from the single-star model, compared with their error bars, scaled so a typical single star
gets 1.)

### 1.1 The 50 parsec sample

Now the real query: every star with parallax above 20 mas (so closer than 50 pc), whose parallax
is measured to better than 10% (`parallax_over_error > 10`), and which has a measured colour.

We also ask for a few quality columns (`phot_bp_rp_excess_factor` and the ones after it).
We'll see why we need them in section 5.

This returns about 45,000 stars and takes a minute or so. A copy of the result is saved in the
`data/` folder, so by default we load that instead. Set `QUERY_ARCHIVE = True` to run the query
yourself.

In [ ]:
QUERY_ARCHIVE = False

query_50pc = '''
SELECT source_id, ra, dec, parallax, parallax_error, pmra, pmdec,
       phot_g_mean_mag, bp_rp, ruwe,
       phot_bp_rp_excess_factor, visibility_periods_used, astrometric_excess_noise,
       phot_bp_mean_flux_over_error, phot_rp_mean_flux_over_error
FROM gaiadr3.gaia_source
WHERE parallax > 20
  AND parallax_over_error > 10
  AND bp_rp IS NOT NULL
'''

if QUERY_ARCHIVE:
    job = Gaia.launch_job_async(query_50pc)
    stars = job.get_results().to_pandas()
else:
    stars = pd.read_csv(DATA_DIR + "gaia_50pc.csv")

print(len(stars), "stars")

## 2. pandas: working with tables

`stars` is a pandas **DataFrame**: a table with named columns, a bit like a spreadsheet. Each
column is basically a numpy array, so everything from the Python basics notebook works on it.

(In the `describe()` output below, the `source_id` numbers are shown in scientific notation
because they are huge. The IDs themselves are fine.)

In [ ]:
stars.head()      # first 5 rows

In [ ]:
stars.describe()  # summary statistics of every column

Get a column with `stars["column_name"]`. Make a new column by assigning to a new name.
Select rows with a boolean mask, just like numpy: `stars[stars["parallax"] > 100]`.

**Try it.** Add a column called `dist_pc` with the distance in parsecs. Then find how many stars
are within 10 pc.

In [ ]:
stars["dist_pc"] = ____    # your code here

within_10pc = stars[____]    # your code here
print(len(within_10pc), "stars within 10 pc")

A quick check: the volume of a sphere grows as $r^3$. If stars are spread evenly around us, there
should be $(50/10)^3 = 125$ times more stars within 50 pc than within 10 pc. Is that roughly
true? Before you look, think about what could make it come out higher or lower. (For example: are
there stars too bright, or too faint, for Gaia to measure? Would that matter more at 10 pc or at
50 pc?) We'll come back to this number.

In [ ]:
print(len(stars) / len(within_10pc))

### 2.1 Sorting

`sort_values` sorts the table by a column. Here are the 10 nearest stars in our sample.

In [ ]:
stars.sort_values("dist_pc").head(10)

The nearest is Proxima Centauri, at 1.30 pc. Notice who is **missing**: Alpha Centauri A and B,
the two bright stars that Proxima orbits. They are about as close (1.34 pc) but are so bright
that they saturate Gaia's detectors, and DR3 doesn't give them a good parallax. The same goes for
Sirius and the other very brightest stars in the sky.

Every catalog has gaps like this. A big part of doing research with a survey is knowing what it
can and can't see.

## 3. Some famous neighbours

Here are the Gaia `source_id`s for some well-known nearby stars:

In [ ]:
famous = {
    "Proxima Centauri": 5853498713190525696,
    "Barnard's Star":   4472832130942575872,
    "Wolf 359":         3864972938605115520,
    "Sirius B":         2947050466531873024,
    "Epsilon Eridani":  5164707970261890560,
    "Tau Ceti":         2452378776434477184,
    "40 Eridani A":     3195919528989223040,
    "40 Eridani B":     3195919254111315712,
    "40 Eridani C":     3195919254111314816,
    "61 Cygni A":       1872046609345556480,
    "61 Cygni B":       1872046574983497216,
}

for name, sid in famous.items():
    row = stars[stars["source_id"] == sid].iloc[0]
    print(f"{name:18s} d = {row['dist_pc']:5.2f} pc   G = {row['phot_g_mean_mag']:5.2f}   BP-RP = {row['bp_rp']:5.2f}")

(`.iloc[0]` picks the first row of the selection. There's exactly one row for each ID.)

A bit about each:

- **Proxima Centauri**: the nearest star to the Sun. A small, cool red dwarf with at least one
  roughly Earth-mass planet in its *habitable zone* (the range of distances from a star where a
  planet could have liquid water on its surface).
- **Barnard's Star**: the star that moves fastest across our sky. Four small planets were found
  around it in 2024 and 2025.
- **Wolf 359**: one of the closest stars, but so dim (a red dwarf about 1/1000 as luminous as
  the Sun) that you need a telescope to see it.
- **Sirius B**: Sirius is the brightest star in the night sky. Sirius B is its tiny companion,
  a **white dwarf**: the leftover core of a dead star, about the size of the Earth. Sirius A is
  too bright for Gaia DR3, but Sirius B is in.
- **Epsilon Eridani**: a young star a bit cooler than the Sun, with a giant planet and a ring of
  dust and rocky debris around it, a bit like our Solar System's asteroid and Kuiper belts.
- **Tau Ceti**: one of the nearest Sun-like stars. In the science-fiction novel
  *Project Hail Mary* (Andy Weir, 2021), it is the star the spaceship *Hail Mary* travels to.
- **40 Eridani**: a triple system. 40 Eridani B was the first white dwarf ever identified, and C is
  a red dwarf. In *Project Hail Mary*, 40 Eridani A is the star orbited by Erid, the home planet
  of the alien character Rocky. (It's also the home of the planet Vulcan in *Star Trek*.)
- **61 Cygni**: a pair of orange dwarfs. In 1838 Friedrich Bessel measured its parallax,
  about 0.3 arcseconds. This was the first time anyone measured the distance to a star.

**Try it.** Barnard's Star is supposed to be the fastest-moving star on the sky. Check this.

1. Compute the total proper motion $\mu = \sqrt{\mu_{\alpha*}^2 + \mu_\delta^2}$ (that is,
   from the `pmra` and `pmdec` columns) for every star, and store it in a new column `pm_total`
   (units: mas/yr).
2. Sort by `pm_total` to find the fastest stars.
3. Is the fastest one Barnard's Star? Compare its `source_id` with `famous["Barnard's Star"]`.
4. How long would it take Barnard's Star to move across the width of the full Moon (about 0.5
   degrees, or 1,800,000 mas)?

In [ ]:
stars["pm_total"] = ____    # your code here

fastest = stars.sort_values(____, ascending=False).head(5)    # your code here
print(fastest[["source_id", "dist_pc", "pm_total", "phot_g_mean_mag"]])

top = fastest.iloc[0]          # .iloc[0] = the first row
print("Fastest is Barnard's Star:", top["source_id"] == ____)    # your code here

barnard_pm = top["pm_total"]
print(f"Years to cross the Moon: {____:.0f}")    # your code here

About 10 arcseconds per year. That sounds fast, but it still takes Barnard's Star almost two
centuries to move one Moon-width. Gaia can measure motions 100,000 times smaller than this.

## 4. A map of the sky

`ra` and `dec` are like longitude and latitude on the sky. Let's just plot them.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(stars["ra"], stars["dec"], s=1, alpha=0.3)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("Dec [deg]")
ax.set_xlim(360, 0)    # astronomers draw RA increasing to the left (as you see it looking up)
plt.show()

This squashes the sphere of the sky onto a rectangle, like a world map that makes Greenland look
huge. A better option is a map projection that keeps areas correct. matplotlib has a few built in.
They want angles in **radians**, with longitude between $-\pi$ and $\pi$.

It's also more natural to use **Galactic coordinates** for this. Galactic longitude $l$ and
latitude $b$ are defined so that the plane of the Milky Way is at $b = 0$ and the Galactic
center is at $l = 0$. `astropy.coordinates.SkyCoord` converts between coordinate systems.

In [ ]:
coords = SkyCoord(ra=stars["ra"].values * u.deg, dec=stars["dec"].values * u.deg, frame="icrs")
gal = coords.galactic

stars["l"] = gal.l.wrap_at(180 * u.deg).deg   # longitude from -180 to 180 instead of 0 to 360
stars["b"] = gal.b.deg

fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(projection="mollweide")
ax.scatter(np.radians(stars["l"]), np.radians(stars["b"]), s=1, alpha=0.3)
ax.grid(True)
ax.set_title("Stars within 50 pc, Galactic coordinates")
plt.show()

(Like RA, Galactic longitude is usually drawn increasing to the left. matplotlib's projections
draw it increasing to the right. We'll ignore that for now.)

The stars are spread fairly evenly over the whole sky. That might surprise you if you've seen
photos of the Milky Way, which is a band across the sky. Most stars in the Milky Way's disk are
within about 300 pc of its middle plane, and the Sun is near that plane. So within 50 pc we're deep
inside the disk, and the stars around us are in every direction.
Go further out and they concentrate toward $b = 0$. You'll see this in the slides.

Two things don't fit that picture, though:

- There are dense clumps near the center of the map ($l \approx 0$, $b \approx 0$). That's the
  direction of the Galactic center, thousands of parsecs away. Why would there be extra stars
  *within 50 pc* in that direction? Keep this in mind for section 5.
- There are faint stripes and arcs. Those come from **Gaia's scanning pattern**: some parts of
  the sky were observed more often than others, so more stars there pass our
  `parallax_over_error > 10` cut.

**Try it.** Add the famous stars to the map as larger red points, with their names next to them.
(`ax.text(x, y, "words")` writes text at position `x, y`. Add a small number to `x`, like 0.03, so
the label doesn't sit right on top of the point. Remember these positions are in radians, not
degrees.)

In [ ]:
fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(projection="mollweide")
ax.scatter(np.radians(stars["l"]), np.radians(stars["b"]), s=1, alpha=0.2, color="gray")

for name, sid in famous.items():
    row = stars[stars["source_id"] == sid].iloc[0]
    x, y = np.radians(row[____]), np.radians(row[____])    # your code here
    ax.scatter(x, y, s=30, color="red")
    ax.text(____, ____, name, fontsize=8)    # your code here

ax.grid(True)
plt.show()

## 5. The Hertzsprung-Russell diagram

The most useful plot in stellar astronomy puts a star's **colour** (which tells you its
temperature) on the x-axis and how much light it really gives off on the y-axis. It's called a
Hertzsprung-Russell (HR) diagram, or a colour-magnitude diagram.

**Magnitudes.** (This uses logarithms: $\log_{10}(x)$ asks "10 to what power gives $x$?")
Astronomers measure brightness in *magnitudes*. If a star sends us a flux $F$
(energy per second per square meter), its magnitude is

$$ m = -2.5 \log_{10} F + \text{constant}. $$

The minus sign is why magnitudes are backwards: brighter stars have *smaller* magnitudes. A
difference of 5 magnitudes is a factor of exactly 100 in flux. (This odd system goes back to the
ancient Greeks, who called the brightest stars "first magnitude".)

The apparent magnitude $G$ tells us how bright a star *looks* in Gaia's G band, which depends on
its distance. To compare stars fairly we use the **absolute magnitude** $M_G$: how bright the star
would look from a standard distance of 10 pc. Flux falls off as $1/d^2$, so moving a star from
distance $d$ to 10 pc changes its magnitude by $2.5 \log_{10}(d^2/10^2) = 5 \log_{10}(d/10)$:

$$ M_G = G - 5 \log_{10}\!\left(\frac{d}{10\,\mathrm{pc}}\right) = G + 5 \log_{10}\!\left(\frac{\varpi}{100\,\mathrm{mas}}\right) $$

(Both forms are the same, because $d = 1000/\varpi$.) We're ignoring dust between us and the
stars, which would make them look fainter. Within 50 pc there's very little dust, so that's fine.

**Try it.** Add a column `abs_G` with the absolute magnitude. Then plot `bp_rp` on the x-axis
and `abs_G` on the y-axis. Because magnitudes are backwards, flip the y-axis so that bright stars
are at the top (`ax.invert_yaxis()`).

In [ ]:
stars["abs_G"] = ____    # your code here

fig, ax = plt.subplots(figsize=(6, 8))
ax.scatter(____, ____, s=1, alpha=0.3, color="k")    # your code here
____    # your code here
ax.set_xlabel(r"$G_{BP} - G_{RP}$ [mag]")
ax.set_ylabel(r"$M_G$ [mag]")
plt.show()

Some of the structure here is real, and some of it isn't. Things you should see:

- **The main sequence**: the long diagonal band from top left (hot, bright, blue) to bottom
  right (cool, faint, red). These are stars burning hydrogen in their cores, like the Sun. Most
  nearby stars are red dwarfs at the lower right, because small stars are much more common than
  big ones.
- **White dwarfs**: the separate, thinner sequence at the lower left. Hot but very faint,
  because they are tiny (about the size of the Earth).
- **Giants**: a few stars at the top right. Cool but very luminous, because they are huge.
  There are only a few, because the giant phase of a star's life is short.

And one thing that shouldn't be there: a big **cloud of points below the main sequence**, around
$M_G \approx 17$. No known kind of star lives there in such numbers.

### 5.1 Something is wrong

Let's find out what these objects are. Select the stars in the cloud and compare them to the
rest.

**Try it.** Select stars with `abs_G > 15.5` and `bp_rp` between 0.8 and 3.5, and call them
`weird`. How many are there? Then compare the median of a few columns between `weird` and
everything else: `phot_g_mean_mag`, `visibility_periods_used` (how many separate times Gaia
looked at the star), and `phot_bp_mean_flux_over_error` (how precisely the blue brightness was
measured). Finally, plot where the `weird` stars are on the sky, using the `l` and `b` columns
you made earlier (converted with `np.radians`, as before).

Reminder: to combine conditions, put each one in parentheses and join them with `&`, like
`(stars["bp_rp"] > 0.8) & (stars["bp_rp"] < 3.5)`. Writing `0.8 < stars["bp_rp"] < 3.5` gives
an error about "truth value of a Series is ambiguous".

In [ ]:
weird_mask = (stars["abs_G"] > 15.5) & (____) & (____)    # your code here
weird = stars[weird_mask]
normal = stars[~weird_mask]
print(len(weird), "weird objects")

for col in ["phot_g_mean_mag", "visibility_periods_used", "phot_bp_mean_flux_over_error"]:
    print(f"{col:30s} weird: {weird[col].median():8.2f}   normal: {____:8.2f}")    # your code here

fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(projection="mollweide")
ax.scatter(____, ____, s=1, color="C3")    # your code here
ax.grid(True)
plt.show()

The weird objects are very faint (around $G = 20$, close to Gaia's limit), were observed fewer
times, have poorly measured colours, and are concentrated toward the Galactic center and the
Galactic plane. These are the most crowded parts of the sky.

What's going on: in crowded fields, faint stars have close neighbours, and Gaia sometimes mixes
up measurements of one star with another. That can produce a large parallax by accident.
These are distant stars with **wrong parallaxes**, not nearby stars. Their colours are wrong too,
because light from the neighbours leaks into the BP and RP measurements.

Most of the time we deal with this using **quality cuts**: conditions that remove sources with
unreliable measurements. The first cut below, on `phot_bp_rp_excess_factor`, comes from the Gaia
team's DR2 HR diagram paper (Gaia Collaboration, Babusiaux et al. 2018). We add a few simple ones
on the number of visits and on how well BP and RP were measured. The `phot_bp_rp_excess_factor` compares the
total flux in BP and RP to the G flux. For a clean, isolated source it follows a known relation
with colour. Contaminated sources have too much flux in BP and RP. (There are more careful
versions of this cut for DR3, but this one does the job here.)

In [ ]:
excess = stars["phot_bp_rp_excess_factor"]
color = stars["bp_rp"]

good = (
    (excess < 1.3 + 0.06 * color**2)
    & (excess > 1.0 + 0.015 * color**2)
    & (stars["visibility_periods_used"] > 10)
    & (stars["phot_bp_mean_flux_over_error"] > 10)
    & (stars["phot_rp_mean_flux_over_error"] > 10)
)
clean = stars[good]
print(f"kept {len(clean)} of {len(stars)} stars")

fig, ax = plt.subplots(1, 2, figsize=(11, 7), sharey=True)
ax[0].scatter(stars["bp_rp"], stars["abs_G"], s=1, alpha=0.3, color="k")
ax[0].set_title("all")
ax[1].scatter(clean["bp_rp"], clean["abs_G"], s=1, alpha=0.3, color="k")
ax[1].set_title("after quality cuts")
ax[0].invert_yaxis()
for a in ax:
    a.set_xlabel(r"$G_{BP} - G_{RP}$ [mag]")
ax[0].set_ylabel(r"$M_G$ [mag]")
plt.show()

Much better. Now go back to the star count test from section 2. **Try it.** With the clean sample,
what is the ratio of stars within 50 pc to stars within 10 pc? How does it compare to 125? Be careful
to count the stars within 10 pc **in `clean`**, not the `within_10pc` table from before.

In [ ]:
ratio = len(clean) / len(clean[____])    # your code here
print(ratio)

Cuts are never perfect. They always throw away some real stars along with the junk, and you
should check what you lost. Which of our famous stars fail the cuts? `~good` means "not good", and
`.isin(...)` checks whether each value is in a list:

In [ ]:
fails = stars[~good & stars["source_id"].isin(famous.values())]
fails[["source_id", "dist_pc", "phot_g_mean_mag", "bp_rp", "phot_bp_rp_excess_factor"]]

It's Sirius B. Why would Sirius B, of all stars, have contaminated colours? (Think about what's
right next to it on the sky.)

Here's why this matters for our project. The BP and RP measurements collect light from a small
window around each star, about 3.5 by 2 arcseconds. Light from any star close by on the sky,
including a binary companion, gets mixed in. So this cut also tends to remove some close binaries.

There's a second trap. Another popular cut is
`astrometric_excess_noise < 1`, which removes stars whose positions scatter more than expected.
It cleans up the diagram even more. But stars with orbiting companions wobble, so they have extra
"noise" too. That cut would quietly throw away many of the binaries and planet hosts we're
looking for. Always ask whether a cut removes the thing you care about.

From now on we'll use `clean`.

### 5.2 Famous stars on the HR diagram

Where is the Sun? It has $G_{BP} - G_{RP} \approx 0.82$ and $M_G \approx 4.67$.

**Try it.** Make the HR diagram of the clean sample again and mark the famous stars on it (and the
Sun). Look the famous stars up in `stars`, not `clean`, so that none go missing. Before you run it,
predict where Sirius B and 40 Eridani B will land.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
ax.scatter(clean["bp_rp"], clean["abs_G"], s=1, alpha=0.2, color="gray")

for name, sid in famous.items():
    row = ____    # your code here
    ax.scatter(____, ____, s=30, color="red")    # your code here
    ax.text(____, ____, name, fontsize=8)    # your code here

ax.scatter(0.82, 4.67, marker="*", s=200, color="gold", edgecolor="k", label="Sun")

ax.invert_yaxis()
ax.set_xlabel(r"$G_{BP} - G_{RP}$ [mag]")
ax.set_ylabel(r"$M_G$ [mag]")
ax.legend()
plt.show()

### 5.3 Hidden binaries

Look closely at the main sequence of the clean sample. There is a fainter, thinner band running
parallel to it, just *above* it.

About half of all Sun-like stars have a companion star. If two stars are too close together for
Gaia to separate, Gaia sees their combined light. Two identical stars together are twice as
bright as one, which is 0.75 magnitudes brighter (since $2.5 \log_{10} 2 \approx 0.75$), at the
same colour. So unresolved twin binaries sit about 0.75 mag above the main sequence. That's the
band.

If the two stars orbit each other with a period of months to years, Gaia also sees the
photocenter wobble. Then the single-star model fits poorly and RUWE goes up. A common rule of
thumb is that `ruwe > 1.4` points to a star that is probably not single.

Before trusting that rule, let's test it on stars we know. **Try it.** Print the RUWE of each of
our famous stars.

In [ ]:
for name, sid in famous.items():
    row = stars[stars["source_id"] == sid].iloc[0]
    print(f"{name:18s} G = {row['phot_g_mean_mag']:5.2f}   RUWE = {____:.2f}")    # your code here

Tau Ceti, Epsilon Eridani and 40 Eridani A all have RUWE well above 1.4, but none of them has a
stellar companion close enough to cause that. What they have in common is that they're **very
bright** ($G < 5$). Gaia's measurements of very bright stars are harder to calibrate, so their
RUWE is inflated. About 40% of stars brighter than $G = 6$ in our sample have RUWE > 1.4.

So RUWE is a useful clue, not a verdict. A few more things to know about it:

- A second star nearby on the sky (but not orbiting closely) can also raise RUWE.
- A wobble with a period much longer than Gaia's observations looks like a straight-line motion,
  and gets absorbed into the proper motion. A period of almost exactly one year looks like
  parallax. Either way, RUWE stays low even though there is a companion.
- The same size of wobble raises RUWE a lot for a bright star, and much less for a faint one.

**Try it.** Now split the clean sample into stars with `ruwe < 1.4` and `ruwe > 1.4`, and plot their HR
diagrams side by side. What fraction of stars have RUWE above 1.4? Compare the two main
sequences. Which one is wider, and in which direction?

In [ ]:
low_ruwe = clean[____]    # your code here
high_ruwe = clean[____]    # your code here
print(f"{len(high_ruwe) / len(clean):.1%} of stars have RUWE > 1.4")

fig, ax = plt.subplots(1, 2, figsize=(11, 7), sharex=True, sharey=True)
ax[0].scatter(____, ____, s=1, alpha=0.2, color="k")    # your code here
ax[0].set_title("RUWE < 1.4")
ax[1].scatter(____, ____, s=1, alpha=0.2, color="C3")    # your code here
ax[1].set_title("RUWE > 1.4")
ax[0].invert_yaxis()
ax[0].set_ylabel(r"$M_G$ [mag]")
for a in ax:
    a.set_xlabel(r"$G_{BP} - G_{RP}$ [mag]")
plt.show()

The high-RUWE main sequence is broader, and spread mostly *upward*: many of those stars are
unresolved binaries, brighter than a single star of the same colour. Notice also that there are
almost no white dwarfs with high RUWE. Two reasons: white dwarfs are faint, and Gaia's measurements
of faint stars are noisy enough to hide a small wobble. And if a white dwarf *does* have a normal
star orbiting close to it, the normal star is usually much brighter, so the pair shows up on the
main sequence, not among the white dwarfs.

## 6. Where are the known exoplanets?

The NASA Exoplanet Archive (https://exoplanetarchive.ipac.caltech.edu/) keeps a table of every
confirmed exoplanet. A copy is in `data/exoplanets.csv` (one row per planet). It includes how
each planet was discovered (`discoverymethod`), its distance (`sy_dist`, in pc), and the Gaia DR3
ID of its host star (`gaia_dr3_id`).

In [ ]:
planets = pd.read_csv(DATA_DIR + "exoplanets.csv")
print(len(planets), "planets")
planets.head()

**Try it.** How many planets were found by each method? pandas has a method for exactly this:
`value_counts()`. Apply it to the `discoverymethod` column.

In [ ]:
planets[____].____()    # your code here

The two big methods:

- **Transit**: the planet passes in front of its star and blocks a little light. Most of these
  come from the Kepler and TESS space telescopes.
- **Radial velocity**: the star wobbles toward and away from us because of the planet, which
  shifts its spectral lines (the Doppler effect).

Radial velocity measures the same wobble that Gaia sees, but along our line of sight instead of
across the sky. That makes the two methods a natural pair. Notice how few planets have been
found by **astrometry** (measuring positions), so far.

Let's see how far away the planets are, for the four most common methods.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bins = np.logspace(0, 4.5, 50)
for method in ["Transit", "Radial Velocity", "Microlensing", "Imaging"]:
    sel = planets["discoverymethod"] == method
    ax.hist(planets.loc[sel, "sy_dist"], bins=bins, histtype="step", lw=2, label=method)
ax.set_xscale("log")
ax.set_xlabel("Distance [pc]")
ax.set_ylabel("Number of planets")
ax.legend()
plt.show()

(`planets.loc[sel, "sy_dist"]` means "rows where `sel` is True, column `sy_dist`".)

Radial velocity planets are mostly nearby (you need a bright star to get a good spectrum).
Transit planets are typically hundreds of pc away. Microlensing finds planets thousands of
parsecs away, toward the center of the Galaxy.

**Try it.** Make a sky map in Galactic coordinates of all the planet hosts, coloured by
discovery method. Use the same four methods. Look back at section 4 for how to get $l$ and $b$
(don't forget `.wrap_at(180 * u.deg)`, or half the sky will come out empty). There will be two
obvious clumps. What are they?

In [ ]:
pl_coords = SkyCoord(ra=planets["ra"].values * u.deg, dec=planets["dec"].values * u.deg)
planets["l"] = ____    # your code here
planets["b"] = ____    # your code here

fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(projection="mollweide")
for method in ["Transit", "Radial Velocity", "Microlensing", "Imaging"]:
    sel = ____    # your code here
    ax.scatter(____, ____, s=4, label=method)    # your code here
ax.grid(True)
ax.legend(loc="lower right", fontsize=8)
plt.show()

<details>
<summary>The two clumps (click to show)</summary>

The dense patch of transit planets near $l \approx 75°$, $b \approx +13°$ is the **Kepler field**.
The Kepler telescope stared at that one patch of sky, in the constellations Cygnus and Lyra, for
four years, and found thousands of planets there.

The microlensing planets are packed near the **Galactic center** ($l \approx 0$, $b \approx 0$),
because microlensing needs lots of background stars, and that's where they are densest.

</details>

### 6.1 Matching planets to Gaia stars

Which of our 50 pc stars host known planets? To combine two tables we need a column they share.
Both have the Gaia DR3 source ID, but in the planet table it's written as text like
`"Gaia DR3 5853498713190525696"`. We need to strip off the `"Gaia DR3 "` and turn the rest into a
number.

pandas has string tools under `.str`. For example, `planets["gaia_dr3_id"].str.replace("A", "B")`
replaces "A" with "B" in every row.

In [ ]:
planets["source_id"] = planets["gaia_dr3_id"].str.replace(____, "")    # your code here
planets["source_id"] = pd.to_numeric(planets["source_id"]).astype("Int64")

planets[["pl_name", "gaia_dr3_id", "source_id"]].head()

(`"Int64"`, with a capital I, is a pandas integer type that allows missing values. A few
planets don't have a Gaia ID.)

Many stars host more than one planet. Let's keep one row per host star, then **merge** with our
Gaia table. `pd.merge(left, right, on="source_id")` keeps only rows whose `source_id` appears in
both tables, and combines their columns.

In [ ]:
# drop_duplicates keeps the first row for each host, so "discoverymethod" is the method of
# that host's first-listed planet. dropna removes rows with a missing source_id.
hosts = planets.drop_duplicates("hostname")[["hostname", "source_id", "discoverymethod", "sy_dist"]]
hosts = hosts.dropna(subset=["source_id"])

nearby_hosts = pd.merge(stars, hosts, on="source_id")
print(len(hosts), "host stars with Gaia IDs,", len(nearby_hosts), "of them in our 50 pc sample")
nearby_hosts[["hostname", "dist_pc", "sy_dist", "discoverymethod"]].sort_values("dist_pc").head(15)

The `dist_pc` column comes from our Gaia parallax and `sy_dist` from the exoplanet archive. They
mostly agree, since the archive mostly uses Gaia distances too. But not always. Let's look at the
biggest disagreements:

In [ ]:
nearby_hosts["dist_diff"] = (nearby_hosts["dist_pc"] - nearby_hosts["sy_dist"]).abs() / nearby_hosts["sy_dist"]
nearby_hosts.sort_values("dist_diff", ascending=False)[["hostname", "dist_pc", "sy_dist", "dist_diff", "phot_g_mean_mag", "ruwe"]].head(6)

The worst cases are off by 5 to 20%. They are bright stars ($G \approx 3$ to 4) with high RUWE,
mostly giants. For these, the archive uses a distance from another source (often the older
Hipparcos satellite). With bright stars and high RUWE, there's a real chance Gaia's parallax is
the one that's off. When two measurements disagree, the quality flags help you decide which to
trust.

**Try it.** Plot the HR diagram one more time, with the planet hosts in our sample highlighted.
What kinds of stars are planet hosts? Are there any white dwarfs or giants among them?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 8))
ax.scatter(clean["bp_rp"], clean["abs_G"], s=1, alpha=0.2, color="gray", label="all stars")
ax.scatter(____, ____, s=10, color="C0", label="planet hosts")    # your code here
ax.invert_yaxis()
ax.set_xlabel(r"$G_{BP} - G_{RP}$ [mag]")
ax.set_ylabel(r"$M_G$ [mag]")
ax.legend()
plt.show()

### 6.2 Planets found by astrometry

Here is the full list of companions that the archive says were found by astrometry. Not all of
them are from Gaia: GJ 896 A b was found with radio telescopes, and DENIS-P J082303.1-491201 b
with a ground-based telescope.

**Gaia-4b** and **Gaia-5b** were among the first discoveries from Gaia's own astrometry. They
started as Gaia DR3 candidates and were confirmed with radial velocities in 2025. Gaia-4b is a
giant planet about 12 times the mass of Jupiter, orbiting a star of 0.64 solar masses every 570
days. Gaia-5b, at about 21 Jupiter masses, is more likely a **brown dwarf** (more on those
below). The archive's "planet" table includes some objects above 13 Jupiter masses. HIP 66074 b
(also called Gaia-3b) was announced earlier, in 2023, but the Gaia team later found that its DR3
orbit was caused by a processing bug (more on this in section 7). The planet itself is still
supported by radial-velocity measurements.

In [ ]:
cols = ["pl_name", "sy_dist", "disc_year", "pl_orbper", "pl_bmassj", "st_mass"]
planets[planets["discoverymethod"] == "Astrometry"][cols]

(`pl_orbper` is the orbital period in days, `pl_bmassj` the planet mass in Jupiter masses, and
`st_mass` the star's mass in solar masses.)

Gaia DR4, scheduled for 2 December 2026, should raise the number of astrometric planets from a
handful to thousands.

## 7. Orbits Gaia has already measured

For most stars, Gaia DR3 fits five numbers: position (2), parallax (1) and proper motion (2).
For about 170,000 stars, Gaia also fitted an **orbit** to the photocenter's motion. These are in a
separate table, `gaiadr3.nss_two_body_orbit` ("nss" means "non-single star"). The
**photocenter** is the center of the light Gaia sees. For a star with a dark companion, that's
just the star.

Gaia only tried to fit orbits for some stars: mainly those with RUWE > 1.4, brighter than
$G = 19$, and observed enough times. Then it only kept orbits that passed strict significance
tests. So this table is far from a complete list of binaries.

The table has a column `nss_solution_type`. We'll use three of the types:

- `Orbital`: orbits found by the standard search, about 135,000 of them.
- `OrbitalTargetedSearch`: orbits fitted for a **pre-selected list** of a few thousand stars that
  were already known or suspected to have planets or brown dwarfs, mostly from radial-velocity
  surveys. The detection thresholds were relaxed for these stars, because astronomers already had a
  reason to look.
- `OrbitalTargetedSearchValidated`: targeted-search orbits that were also checked against
  published radial velocities and found to agree.

(Other types include orbits combined with radial velocities from Gaia's spectrometer, which we're
leaving out for simplicity.)

We saved these orbits for stars within 100 pc.

Here's the query, if you want to run it yourself:

In [ ]:
query_orbits = '''
SELECT o.source_id, o.nss_solution_type, o.period, o.period_error, o.eccentricity,
       o.a_thiele_innes, o.b_thiele_innes, o.f_thiele_innes, o.g_thiele_innes,
       o.parallax, o.parallax_error, o.significance, g.phot_g_mean_mag, g.bp_rp
FROM gaiadr3.nss_two_body_orbit AS o
JOIN gaiadr3.gaia_source AS g ON o.source_id = g.source_id
WHERE o.nss_solution_type IN ('Orbital', 'OrbitalTargetedSearch', 'OrbitalTargetedSearchValidated')
  AND o.parallax > 10
'''

if QUERY_ARCHIVE:
    orbits = Gaia.launch_job_async(query_orbits).get_results().to_pandas()
else:
    orbits = pd.read_csv(DATA_DIR + "gaia_dr3_orbits_100pc.csv")

print(len(orbits), "orbits")
orbits.head()

(The `JOIN` combines two tables inside the archive, the same idea as `pd.merge`.)

`period` is in days. The size of the orbit isn't stored directly. Instead, Gaia gives four
**Thiele-Innes constants** $A, B, F, G$ (in mas), which describe the size, shape and orientation of
the orbit's ellipse on the sky in a way that's convenient for fitting. Roughly, $(A, B)$ and
$(F, G)$ are the sky positions of two points on the ellipse, measured from its center: the end of
the long axis and a point on the short axis, before the ellipse is stretched by tilting. The angular semi-major axis
of the photocenter's orbit, $a_0$, can be recovered from them with:

$$ u = \frac{A^2 + B^2 + F^2 + G^2}{2}, \qquad v = AG - BF, \qquad a_0 = \sqrt{u + \sqrt{u^2 - v^2}} $$

(Notice $G$ here is a Thiele-Innes constant, not the G magnitude.)

**Try it.** Write a function that does this, and use it to add a column `a0_mas`. Then convert it to
AU: an angle in mas divided by the parallax in mas gives a length in AU. (This follows from the
definition of the parsec: at a distance where 1 AU looks like $\varpi$, an orbit that looks like
$a_0$ must be $a_0 / \varpi$ AU across.)

In [ ]:
def thiele_innes_to_a0(A, B, F, G):
    u = ____    # your code here
    v = ____    # your code here
    return ____    # your code here

orbits["a0_mas"] = thiele_innes_to_a0(orbits["a_thiele_innes"], orbits["b_thiele_innes"],
                                      orbits["f_thiele_innes"], orbits["g_thiele_innes"])
orbits["a0_au"] = ____    # your code here
orbits["period_yr"] = ____    # your code here

orbits[["source_id", "period_yr", "a0_mas", "a0_au"]].describe()

### 7.1 Planet or star?

How big would $a_0$ be for a planet? If the companion is dark, the photocenter is just the
visible star, so (as in the Python basics notebook) $a_0 = a\, m_c / (M_\star + m_c)$, where $a$ is the
separation between the two bodies. Kepler's third law gives $a$ from the period:
$a^3 = (M_\star + m_c)\, P^2$, with $a$ in AU, masses in solar masses, and $P$ in years.

Most stars within 100 pc are smaller than the Sun, so let's draw lines on the period vs. $a_0$
plane for companions of different masses around a star of 0.6 solar masses.

In [ ]:
def a0_dark_companion(P_yr, m_comp, m_star):
    '''Photocenter orbit size [AU] for a dark companion. Masses in solar masses.'''
    m_tot = m_star + m_comp
    a = (m_tot * P_yr**2) ** (1 / 3)
    return a * m_comp / m_tot

from astropy import constants as const
m_jup = (const.M_jup / const.M_sun).value   # Jupiter mass in solar masses, about 1/1048
P_grid = np.logspace(-1.5, 1.5, 100)

fig, ax = plt.subplots(figsize=(8, 6))
for name, sel in orbits.groupby("nss_solution_type"):
    ax.scatter(sel["period_yr"], sel["a0_au"], s=5, label=name)

for mc, lab in [(1 * m_jup, r"1 $M_J$"), (13 * m_jup, r"13 $M_J$"), (0.08, r"0.08 $M_\odot$")]:
    ax.plot(P_grid, a0_dark_companion(P_grid, mc, 0.6), "k--", lw=1)
    ax.text(P_grid[-1], a0_dark_companion(P_grid[-1], mc, 0.6), " " + lab, va="center", fontsize=9)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Period [years]")
ax.set_ylabel(r"Photocenter semi-major axis $a_0$ [AU]")
ax.legend(fontsize=8)
plt.show()

(`groupby` splits the table by the values in a column. Each time round the loop, `name` is one
solution type and `sel` holds the rows of that type.)

The lines mark companions of 1 and 13 Jupiter masses ($M_J$), and 0.08 solar masses ($M_\odot$).
Objects above about 13 Jupiter masses are hot enough inside to fuse deuterium (a heavy form of
hydrogen) for a while, and are usually not called planets. Above 0.08 solar masses (about 80
Jupiter masses) they fuse ordinary hydrogen and are stars. Between the two are **brown dwarfs**.

Almost all the points are binary stars. A few sit near or below the 13 $M_J$ line: those are
Gaia **planet candidates**. Notice also the gap in the points near a period of 1 year. A wobble
with a 1-year period looks just like parallax, so Gaia struggles to find those orbits.

### 7.2 A better mass for each star

The lines above assume every star has 0.6 solar masses. But the line for a planet depends on the
star's mass: for the same period, a smaller star wobbles more. Let's estimate each star's mass
from its absolute magnitude. For main-sequence stars there's a tight relation between the two.
Here are rough values:

In [ ]:
# Rough absolute G magnitude and mass for main-sequence stars, from F-type stars down to late M
# dwarfs (approximate values based on E. Mamajek's table of dwarf star properties).
table_MG   = np.array([3.26, 4.10, 4.64, 5.55, 6.83, 8.16, 9.29, 10.05, 11.21, 12.45, 14.26])
table_mass = np.array([1.33, 1.13, 1.00, 0.88, 0.70, 0.57, 0.44, 0.37, 0.23, 0.162, 0.102])

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(table_MG, table_mass, "o-")
ax.set_xlabel(r"$M_G$ [mag]")
ax.set_ylabel(r"Mass [$M_\odot$]")
plt.show()

`np.interp(x, table_x, table_y)` looks up each value of `x` in a table, and draws straight lines
between the table's points. (For values past the ends of the table, it uses the end values.)

**Try it.**

1. Add a column `abs_G` to `orbits` (same formula as in section 5, using the `parallax` column of
   `orbits`), and a column `mass_est` using `np.interp`.
2. Select the orbits that fall below the 13 Jupiter-mass line **for each star's own mass**: call
   `a0_dark_companion` with each orbit's period (`orbits["period_yr"]`), a companion mass of
   `13 * m_jup` (remember, masses are in solar masses), and `orbits["mass_est"]`.
3. Count the candidates, and count how many of each `nss_solution_type` there are among them.

In [ ]:
orbits["abs_G"] = ____    # your code here
orbits["mass_est"] = np.interp(____, table_MG, table_mass)    # your code here

below = orbits["a0_au"] < a0_dark_companion(____, ____, ____)    # your code here
candidates = orbits[below]
print(len(candidates), "orbits below the 13 Jupiter-mass line")
candidates["nss_solution_type"].value_counts()

**Try it.** Sort `candidates` by `abs_G` and look at the `nss_solution_type`, `abs_G` and
`mass_est` columns. Do you notice two groups?

In [ ]:
candidates.sort_values(____)[["source_id", "nss_solution_type", "period_yr", "a0_au", "abs_G", "mass_est"]]    # your code here

The candidates split into two groups:

1. **Sun-like stars** ($M_G \approx 3.5$ to 7) that almost all come from the **targeted search**.
   Many of these already had planets known from radial velocities. They're here partly because Gaia
   was specifically told to look at them, with relaxed thresholds. That's a *selection effect*:
   the catalog shows us these orbits because of how it was made, not because they're common.
2. **Red dwarfs** ($M_G \gtrsim 9$) from the standard search, with periods of about 1 to 2
   years. These would be planets of several Jupiter masses around small stars, if the companions
   are dark.

Group 2 is exactly where impostors are expected to be most common (more on this below). Whenever
you see a pattern in a catalog, ask how the catalog was made.

**Try it.** Which of the candidates are known planet hosts in the NASA table? Merge `candidates`
with the `planets` table (not `hosts`, because we want each planet's name and mass) on
`source_id`.

In [ ]:
known = pd.merge(candidates, ____, on="source_id")    # your code here
known[["pl_name", "nss_solution_type", "period_yr", "a0_au", "mass_est", "pl_bmassj", "discoverymethod"]]

Some of these are planets and brown dwarfs that were already known from radial velocities, which
Gaia has now seen in astrometry too. Others are not in the NASA table.

Some candidates deserve a warning. The Gaia team has found that four orbits in DR3 are not real.
Let's see which of them made it into our list:

In [ ]:
retracted = {
    "WD 0141-675": 4698424845771339520,
    "HIP 64690":   5765846127180770432,
    "54 Cas":      522135261462534528,
    "HIP 66074":   1712614124767394816,
}
for name, sid in retracted.items():
    print(f"{name:12s} in orbits table: {sid in orbits['source_id'].values}   in candidates: {sid in candidates['source_id'].values}")

WD 0141-675 is a white dwarf. Its Gaia orbit suggested a giant planet with a 34-day period, and
it got a lot of attention. Then radial-velocity observations found nothing, and in May 2023 the
Gaia team announced that the orbit was caused by a bug in the processing software. In 2024, while
preparing DR4, they found the same bug had produced false orbits for three more stars, including
54 Cas and HIP 66074 from our candidate list. (HIP 66074 does seem to have a planet, found with
radial velocities, but Gaia's DR3 orbit for it is not trustworthy.)

These rows are all still in the DR3 table, because published catalogs are never changed after
release. Catalogs contain mistakes, and checking candidates one by one is part of the job. (Also,
our mass estimate is wrong for WD 0141-675: our table only works for main-sequence stars, and
white dwarfs are much heavier than their brightness suggests.)

### 7.3 The catch

All of this assumed the companion is **dark**. Now think back to the question at the end of
the Python basics notebook. If the companion is a star, it contributes light too, and the photocenter sits
somewhere between the two stars. The photocenter orbit then gets *smaller*. For two identical
stars it vanishes completely.

So a pair of *nearly* identical stars, with a period of a year or two, can produce a photocenter
orbit just as small as a giant planet's. From the astrometry alone, the two can look the same.
These are called **impostors**: binary stars that look like planets.

This is not a hypothetical problem. When astronomers followed up Gaia DR3 planet and brown dwarf
candidates with radial velocities, many turned out to be near-twin binaries. In one study
published in 2026, 13 out of 20 candidates were impostors. Forecasts for DR4 say impostors will be
rarest around Sun-like stars, and most common around red dwarfs with orbits smaller than about
1 AU, where roughly half of the candidates may be impostors. That's group 2 above.

You might think the HR diagram could catch them, since a twin binary is 0.75 mag brighter than a
single star. It helps, but less than you'd hope. Stars of the same colour already differ in
brightness because of age and composition. In simulations of future Gaia data, an HR diagram cut
removed only about 20% of impostors (while also losing about 5% of real planets). An impostor also fools our mass estimate: we think the star
is brighter, so more massive, than either star really is.

Gaia DR4 will contain thousands of planet candidates. Working out which ones are really planets,
and which are binary stars in disguise, is the problem we'll work on in this project.

## pandas summary

| Code | What it does |
|---|---|
| `pd.read_csv("file.csv")` | Load a table |
| `df.head()`, `df.describe()`, `len(df)` | First rows, summary statistics, number of rows |
| `df["col"]` | One column |
| `df["new"] = ...` | Add a column |
| `df[df["col"] > 5]` | Keep rows where a condition is true |
| `df.sort_values("col", ascending=False)` | Sort |
| `df.iloc[0]` | First row |
| `df.loc[mask, "col"]` | Column `col`, only rows where `mask` is true |
| `df["col"].value_counts()` | Count each distinct value |
| `df["col"].isin(list)` | True where the value is in the list |
| `df["col"].str.replace("a", "b")` | Replace text in every row |
| `df.drop_duplicates("col")`, `df.dropna(subset=["col"])` | Remove repeated rows, rows with missing values |
| `pd.merge(df1, df2, on="col")` | Combine two tables using a shared column |
| `for name, group in df.groupby("col")` | Loop over groups of rows with the same value |

## Things to try on your own

- Repeat the HR diagram for stars within 20 pc only. Which features get clearer, and which get
  harder to see?
- Colour the 50 pc sky map by `bp_rp`. Are red and blue stars spread the same way across the sky?
- Use `parallax_error` and `phot_g_mean_mag` to plot how Gaia's precision depends on brightness.
  Use a log y-axis. Can you explain the shape? (There's a jump near $G = 13$. It comes from a
  change in how Gaia's camera reads out stars of different brightness.)
- Instead of the single ratio from section 2, plot the number of clean stars within distance $d$
  against $d$, on log-log axes. What slope do you expect? What do you get?
- Measure the binary sequence: for stars with $1.5 < G_{BP} - G_{RP} < 2.5$, fit a straight line
  to the main sequence, then make a histogram of how far each star sits above or below the line.
  Can you see a second bump about 0.75 mag above?
- Look up one of the candidates from section 7.2 on SIMBAD (https://simbad.cds.unistra.fr/simbad/)
  using its Gaia DR3 ID. What is known about it?
- Write your own ADQL query. For example: all stars within 100 pc brighter than G = 6.